# Advanced Analytics — Bluestock MF Capstone

This notebook completes the outstanding D6/advanced-analytics requirements using the actual project SQLite database.

### Required tasks
1. Historical VaR 95% and CVaR for all schemes
2. Rolling 90-day Sharpe for five key funds
3. Investor cohorts by first transaction year
4. SIP continuity and >35-day at-risk flag
5. Low / Moderate / High risk-appetite recommender
6. Sector HHI concentration for equity funds
7. Five advanced written insights

### Outputs
- `data/processed/var_cvar_report.csv`
- `data/processed/investor_cohort_first_year.csv`
- `data/processed/sip_continuity_analysis.csv`
- `data/processed/sector_hhi_equity_funds.csv`
- `scripts/recommender.py`
- `reports/rolling_sharpe_chart.png`


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DB_PATH = PROJECT_ROOT / "bluestock_mf.db"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
for d in [PROCESSED_DIR, REPORTS_DIR, SCRIPTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def load_table(name):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(f'SELECT * FROM "{name}"', conn)

fund = load_table("dim_fund")
nav = load_table("fact_nav")
tx = load_table("fact_transactions")
portfolio = load_table("fact_portfolio")
perf = load_table("fact_performance")

nav["date"] = pd.to_datetime(nav["date"], errors="coerce")
nav["daily_return_pct"] = pd.to_numeric(nav["daily_return_pct"], errors="coerce")
nav["daily_return"] = nav["daily_return_pct"] / 100
tx["date"] = pd.to_datetime(tx["date"], errors="coerce")
tx["amount"] = pd.to_numeric(tx["amount"], errors="coerce")
portfolio["weight_pct"] = pd.to_numeric(portfolio["weight_pct"], errors="coerce")
portfolio["as_of_date"] = pd.to_datetime(portfolio["as_of_date"], errors="coerce")

print("Funds:", len(fund))
print("NAV rows:", len(nav))
print("Transactions:", len(tx))
print("Portfolio rows:", len(portfolio))


## 1. Historical VaR (95%) and CVaR

In [ ]:
var_rows = []

for amfi_code, g in nav.dropna(subset=["amfi_code", "daily_return"]).groupby("amfi_code"):
    r = g.sort_values("date")["daily_return"].dropna()
    if len(r) < 20:
        continue

    threshold = r.quantile(0.05)
    var95 = -threshold
    tail = r[r <= threshold]
    cvar95 = -tail.mean()

    var_rows.append({
        "amfi_code": amfi_code,
        "observations": len(r),
        "var_95_pct": var95 * 100,
        "cvar_95_pct": cvar95 * 100
    })

var_cvar = pd.DataFrame(var_rows).merge(
    fund[["amfi_code", "scheme_name", "fund_house", "category", "risk_grade"]],
    on="amfi_code",
    how="left"
).sort_values("var_95_pct", ascending=False)

var_cvar.to_csv(
    PROCESSED_DIR / "var_cvar_report.csv",
    index=False
)

print("Schemes with VaR/CVaR:", len(var_cvar))
display(var_cvar)


**Formula:** VaR is the negative 5th percentile of daily returns. CVaR is the negative mean of returns at or below the 5th-percentile threshold. Positive values therefore represent loss magnitude.

## 2. Rolling 90-day Sharpe — five key funds

In [ ]:
nav_roll = (
    nav.dropna(subset=["amfi_code", "date", "daily_return"])
    .sort_values(["amfi_code", "date"])
    .copy()
)

nav_roll["rolling_90_sharpe"] = (
    nav_roll.groupby("amfi_code")["daily_return"]
    .transform(
        lambda s: (
            s.rolling(90, min_periods=90).mean()
            / s.rolling(90, min_periods=90).std()
            * np.sqrt(252)
        )
    )
)

key_codes = (
    perf.sort_values("return_1yr_pct", ascending=False)
    ["amfi_code"]
    .head(5)
    .tolist()
)

rolling = nav_roll[nav_roll["amfi_code"].isin(key_codes)].merge(
    fund[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
)

plt.figure(figsize=(13, 7))
for code, g in rolling.groupby("amfi_code"):
    g = g.sort_values("date")
    plt.plot(g["date"], g["rolling_90_sharpe"], label=g["scheme_name"].iloc[0][:45])

plt.axhline(0, linewidth=1)
plt.title("Rolling 90-Day Sharpe Ratio — Five Key Funds")
plt.xlabel("Date")
plt.ylabel("Rolling 90-Day Sharpe")
plt.grid(alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    REPORTS_DIR / "rolling_sharpe_chart.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()


## 3. Investor cohorts by first transaction year

In [ ]:
txc = tx.dropna(subset=["investor_id", "date", "amount"]).copy()

first_transaction = (
    txc.groupby("investor_id")["date"]
    .min()
    .rename("first_transaction_date")
)

txc = txc.merge(first_transaction, on="investor_id", how="left")
txc["cohort_year"] = txc["first_transaction_date"].dt.year

is_sip = txc["transaction_type"].astype(str).str.contains(
    "sip", case=False, na=False
)

sip_txc = txc[is_sip].copy()

cohort = (
    txc.groupby("cohort_year")
    .agg(
        investors=("investor_id", "nunique"),
        total_invested=("amount", "sum"),
        avg_transaction_amount=("amount", "mean")
    )
    .reset_index()
)

sip_summary = (
    sip_txc.groupby("cohort_year")
    .agg(
        sip_transactions=("tx_id", "count"),
        avg_sip_amount=("amount", "mean")
    )
    .reset_index()
)

preference = (
    txc.groupby(["cohort_year", "amfi_code"])
    .agg(transaction_count=("tx_id", "count"))
    .reset_index()
    .sort_values(
        ["cohort_year", "transaction_count"],
        ascending=[True, False]
    )
    .drop_duplicates("cohort_year")
)

preference = preference.merge(
    fund[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
).rename(columns={
    "amfi_code": "top_fund_amfi_code",
    "scheme_name": "top_fund"
})

cohort = cohort.merge(sip_summary, on="cohort_year", how="left")
cohort = cohort.merge(
    preference[["cohort_year", "top_fund_amfi_code", "top_fund"]],
    on="cohort_year",
    how="left"
)

cohort.to_csv(
    PROCESSED_DIR / "investor_cohort_first_year.csv",
    index=False
)

display(cohort)


## 4. SIP continuity — 6+ transactions and >35-day at-risk flag

In [ ]:
sip_dates = txc[is_sip].copy()

continuity = []

for investor_id, g in sip_dates.groupby("investor_id"):
    dates = np.sort(g["date"].dt.normalize().dropna().unique())

    if len(dates) < 6:
        continue

    gaps = np.diff(dates).astype("timedelta64[D]").astype(int)

    avg_gap = gaps.mean()
    max_gap = gaps.max()

    continuity.append({
        "investor_id": investor_id,
        "sip_transactions": len(dates),
        "first_sip_date": pd.Timestamp(dates[0]),
        "last_sip_date": pd.Timestamp(dates[-1]),
        "avg_gap_days": avg_gap,
        "max_gap_days": max_gap,
        "at_risk": avg_gap > 35
    })

continuity = pd.DataFrame(continuity)

continuity.to_csv(
    PROCESSED_DIR / "sip_continuity_analysis.csv",
    index=False
)

if not continuity.empty:
    at_risk_rate = continuity["at_risk"].mean() * 100
    print(f"Investors with 6+ SIP transactions: {len(continuity):,}")
    print(f"At-risk rate: {at_risk_rate:.2f}%")
    display(continuity.head(20))
else:
    print("No investors met the 6+ SIP transaction threshold.")


## 5. Simple risk-appetite fund recommender

In [ ]:
def recommend_funds(risk_appetite, top_n=3):
    risk_appetite = risk_appetite.strip().title()

    if risk_appetite not in {"Low", "Moderate", "High"}:
        raise ValueError("Use Low, Moderate, or High.")

    result = (
        fund.merge(
            perf[["amfi_code", "sharpe_ratio", "return_1yr_pct",
                  "std_dev_pct", "max_drawdown_pct"]],
            on="amfi_code",
            how="inner"
        )
    )

    result = result[
        result["risk_grade"].astype(str).str.strip().str.lower()
        == risk_appetite.lower()
    ]

    return (
        result.sort_values("sharpe_ratio", ascending=False)
        .head(top_n)
        [[
            "scheme_name",
            "fund_house",
            "category",
            "risk_grade",
            "sharpe_ratio",
            "return_1yr_pct",
            "std_dev_pct",
            "max_drawdown_pct"
        ]]
    )

for appetite in ["Low", "Moderate", "High"]:
    print(f"\n{appetite} risk recommendations")
    display(recommend_funds(appetite))


The standalone `scripts/recommender.py` accepts `Low`, `Moderate`, or `High` and returns the top 3 schemes by Sharpe ratio within the matching `risk_grade`.

## 6. Sector HHI concentration

In [ ]:
equity_codes = set(
    fund.loc[
        fund["category"].astype(str).str.contains("equity", case=False, na=False),
        "amfi_code"
    ].dropna()
)

# If category labels do not contain 'Equity', portfolio-covered funds are used as fallback.
if not equity_codes:
    equity_codes = set(portfolio["amfi_code"].dropna().unique())

sector = (
    portfolio[
        portfolio["amfi_code"].isin(equity_codes)
    ]
    .dropna(subset=["amfi_code", "sector", "weight_pct"])
    .groupby(["amfi_code", "sector"], as_index=False)["weight_pct"]
    .sum()
)

sector["weight_fraction"] = sector["weight_pct"] / 100

hhi = (
    sector.groupby("amfi_code")["weight_fraction"]
    .apply(lambda s: (s ** 2).sum())
    .rename("sector_hhi")
    .reset_index()
)

hhi["sector_hhi_10000"] = hhi["sector_hhi"] * 10000

hhi = hhi.merge(
    fund[["amfi_code", "scheme_name", "fund_house", "category"]],
    on="amfi_code",
    how="left"
).sort_values("sector_hhi", ascending=False)

hhi.to_csv(
    PROCESSED_DIR / "sector_hhi_equity_funds.csv",
    index=False
)

display(hhi)


### HHI interpretation

HHI = Σ(weightᵢ²), where weights are expressed as fractions.

- HHI near 0: diversified across many sectors
- Higher HHI: more concentrated portfolio
- Conventional HHI presentation multiplies the fraction-based result by 10,000

The calculation is performed at the sector level and is therefore a sector-concentration measure, not a stock-level concentration measure.

## 7. Five advanced insights

In [ ]:
insights = []

if not var_cvar.empty:
    row = var_cvar.iloc[0]
    insights.append(
        f"**1. Highest tail risk:** {row['scheme_name']} has the highest observed "
        f"95% historical VaR at {row['var_95_pct']:.2f}%, with CVaR of {row['cvar_95_pct']:.2f}%."
    )

if not cohort.empty:
    row = cohort.sort_values("total_invested", ascending=False).iloc[0]
    insights.append(
        f"**2. Largest investor cohort by capital:** the {int(row['cohort_year'])} first-transaction "
        f"cohort invested ₹{row['total_invested']:,.0f} in the modeled transactions."
    )

if not continuity.empty:
    rate = continuity["at_risk"].mean() * 100
    insights.append(
        f"**3. SIP continuity:** {len(continuity):,} investors have at least six SIP transactions; "
        f"{rate:.1f}% have an average SIP gap above 35 days and are flagged at-risk."
    )

if not hhi.empty:
    row = hhi.iloc[0]
    insights.append(
        f"**4. Highest sector concentration:** {row['scheme_name']} has the highest sector HHI "
        f"at {row['sector_hhi_10000']:.0f} on the conventional 0–10,000 scale."
    )

if not var_cvar.empty:
    insights.append(
        f"**5. Tail-risk benchmark:** the median historical 95% VaR across the analyzed schemes is "
        f"{var_cvar['var_95_pct'].median():.2f}%; funds above this level have higher observed "
        f"one-day tail-loss magnitude than the median scheme."
    )

for text in insights:
    print(text)


## Final output checklist

| Requirement | Output |
|---|---|
| VaR + CVaR | `data/processed/var_cvar_report.csv` |
| Rolling 90-day Sharpe | `reports/rolling_sharpe_chart.png` |
| Investor first-year cohorts | `data/processed/investor_cohort_first_year.csv` |
| SIP continuity / at-risk | `data/processed/sip_continuity_analysis.csv` |
| Risk-appetite recommender | `scripts/recommender.py` |
| Sector HHI | `data/processed/sector_hhi_equity_funds.csv` |
| Five advanced insights | This notebook |

**Financial disclaimer:** All metrics and recommendations are historical analytical outputs for the capstone. They are not personalized financial advice or guarantees of future performance.
